In [ ]:
import sys
import os
import math
import numpy as np

states = { "s": 0, "E": 1, "5": 2, "I" : 3, "e": 4}
id2state = {0: "s", 1: "E", 2: "5", 3: "I", 4: "e"}

state_transition_prob = np.array([[0.0, 1.0, 0.0, 0.0, 0.0], 
                                  [0.0, 0.9, 0.1, 0.0, 0.0], 
                                  [0.0, 0.0, 0.0, 1.0, 0.0],
                                  [0.0, 0.0, 0.0, 0.9, 0.1],
                                  [0.0, 0.0, 0.0, 0.0, 0.0]]) 
emission_nuc_codes = {'A': 0, 
                      'C': 1, 
                      'G': 2, 
                      'T': 3}

emission_probs = np.array([[0.00, 0.00, 0.00, 0.00], 
                           [0.25, 0.25, 0.25, 0.25],
                           [0.05, 0.00, 0.95, 0.00],
                           [0.40, 0.10, 0.10, 0.40],
                           [0.00, 0.00, 0.00, 0.00]]) 

query_sequence = "CTTCATGTGAAAGCAGACGTAAGTCA"


In [ ]:
def get_log_prob_for_state_path(state_path, query_sequence):
    res = math.log(0.25)
    for i in range(1, len(state_path)):
        res += math.log(state_transition_prob[ states[state_path[i-1]] ][ states[state_path[i]] ] * emission_probs[ states[state_path[i]] ][ emission_nuc_codes[query_sequence[i]] ])
    return res

In [ ]:
# CTTCATGTGAAAGCAGACGTAAGTCA 
# EEEEEE5IIIIIIIIIIIIIIIIIII
k1 = get_log_prob_for_state_path("EEEEEE5IIIIIIIIIIIIIIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log(0.1)
print(k1)


-43.89740030179307


In [ ]:
# CTTCATGTGAAAGCAGACGTAAGTCA 
# EEEEEEEE5IIIIIIIIIIIIIIIII
k2 = get_log_prob_for_state_path("EEEEEEEE5IIIIIIIIIIIIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log(0.1)
print(k2)


In [ ]:
# CTTCATGTGAAAGCAGACGTAAGTCA 
# EEEEEEEEEEEE5IIIIIIIIIIIII
k3 = get_log_prob_for_state_path("EEEEEEEEEEEE5IIIIIIIIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log(0.1)
print(k3)


In [ ]:
# CTTCATGTGAAAGCAGACGTAAGTCA 
# EEEEEEEEEEEEEEE5IIIIIIIIII
k4 = get_log_prob_for_state_path("EEEEEEEEEEEEEEE5IIIIIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log(0.1)
print(k4)


In [ ]:
# CTTCATGTGAAAGCAGACGTAAGTCA 
# EEEEEEEEEEEEEEEEEE5IIIIIII
k5 = get_log_prob_for_state_path("EEEEEEEEEEEEEEEEEE5IIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log(0.1)
print(k5)


In [ ]:
# CTTCATGTGAAAGCAGACGTAAGTCA 
# EEEEEEEEEEEEEEEEEEEEEE5III
k6 = get_log_prob_for_state_path("EEEEEEEEEEEEEEEEEEEEEE5III", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log(0.1)
print(k6)


In [ ]:
# CTTCATGTGAAAGCAGACGTAAGTCA 
# EEEEEEEEEEEEEEEEEEEEEEEEEE
only_E = get_log_prob_for_state_path("EEEEEEEEEEEEEEEEEEEEEEEEEE", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log(0.1)
print(only_E)


### Design of the Viterbi Value matrix

Rows correspond to the hidden states, and the columns correspond to the emissions that is the observed nucleotide sequences. Here I am showing the calculation for the first two nucletides. 

```
             C                                                          T     T
s [s-s-C(0.00) max(s-s-C-s-T, s-E-C-s-T, s-5-C-s-T, s-I-C-s-T, s-e-C-s-T)     .] 
E [s-E-C(0.25) max(s-s-C-E-T, s-E-C-E-T, s-5-C-E-T, s-I-C-E-T, s-e-C-E-T)     .] 
5 [s-5-C(0.00) max(s-s-C-5-T, s-E-C-5-T, s-5-C-5-T, s-I-C-5-T, s-e-C-5-T)     .]
I [s-I-C(0.00) max(s-s-C-I-T, s-E-C-I-T, s-5-C-I-T, s-I-C-I-T  s-e-C-I-T)     .]
e [s-e-C(0.00) max(s-s-C-e-T, s-E-C-e-T, s-5-C-e-T, s-I-C-e-T, s-e-C-e-T)     .]

```

It is important to remember that you will be working in the log scale.

In [ ]:
num_states = len(states)      # 5 states: s, E, 5, I, e
seq_len = len(query_sequence) # 26 nucleotides

# -inf represents log(0), i.e. an impossible/unreachable state
viterbi_value_matrix = np.full((num_states, seq_len), -np.inf)
viterbi_trace_matrix = np.zeros((num_states, seq_len), dtype=int)

# initialize the first column using transitions from the start state (index 0)
# log(0.25) is the initial prob - uniform prior over the 4 possible starting nucleotides
for cur_state in range(num_states):
    nuc_idx = emission_nuc_codes[query_sequence[0]]
    emission = emission_probs[cur_state][nuc_idx]        # BUG FIX: was 'emmision' (typo)
    transition = state_transition_prob[0][cur_state]
    if emission > 0 and transition > 0:
        viterbi_value_matrix[cur_state][0] = math.log(0.25) + math.log(transition) + math.log(emission)
    viterbi_trace_matrix[cur_state][0] = 0  # all paths in col 0 come from start


### Implementation of Viterbi Algorithm
Write a function `calculate_prob_for_a_node()` that populate a single cell in the matrix. The function will return two values:
1. the maximum value, for example, look at the 2nd row, 2nd column in the matrix: `max(s-s-C-E-T, s-E-C-E-T, s-5-C-E-T, s-I-C-E-T, s-e-C-E-T)`. If the probability for `s-E-C-E-T` is highest (lets say X), then the function should return `X`

**AND** 

2. The index of that maximum value described in the first point: so index of X is `1` (recall that Python works on the 0-based index system)

- Populate `viterbi_value_matrix` with `X` for row 2 and col 2

- Populate `viterbi_trace_matrix` with `1` for row 2 and col 2

In [ ]:
def calculate_prob_for_a_node(prev_col, curr_state, obs_nuc):
    """
    Finds the best log probability for reaching curr_state at the current
    position, given the viterbi values from the previous column (prev_col).
    Also returns which previous state gave that best value (for traceback).
    """
    nuc_idx = emission_nuc_codes[obs_nuc]
    emission = emission_probs[curr_state][nuc_idx]

    # if this state can't emit the current nucleotide, there's no point continuing
    if emission == 0:
        return -np.inf, 0

    log_emission = math.log(emission)

    best_val = -np.inf
    best_prev_state = 0

    # try every possible previous state and pick the one giving max probability
    for prev_state in range(num_states):
        transition = state_transition_prob[prev_state][curr_state]

        if transition == 0:
            continue  # this transition doesn't exist in our model

        if prev_col[prev_state] == -np.inf:
            continue  # previous state was unreachable, skip it

        val = prev_col[prev_state] + math.log(transition) + log_emission

        if val > best_val:
            best_val = val
            best_prev_state = prev_state

    return best_val, best_prev_state


In [ ]:
# go column by column from position 1 onwards
# (col 0 was already handled during initialization)
for col in range(1, seq_len):
    for row in range(num_states):
        best_val, best_prev_state = calculate_prob_for_a_node(
            viterbi_value_matrix[:, col - 1],  # values from the previous position
            row,                                # state we're computing for
            query_sequence[col]                 # nucleotide observed at this position
        )
        viterbi_value_matrix[row][col] = best_val
        viterbi_trace_matrix[row][col] = best_prev_state

print("Viterbi value matrix filled.")


Viterbi value matrix filled.


In [ ]:
def traceback(viterbi_value_matrix, viterbi_trace_matrix):
    """
    Walks back through viterbi_trace_matrix to reconstruct
    the most probable hidden state sequence.
    """
    last_col = viterbi_value_matrix[:, -1]

    # the best final state is whichever has the highest value in the last column
    best_last_state = int(np.argmax(last_col))
    print(f"Best log probability: {last_col[best_last_state]:.5f}")

    path = [best_last_state]
    curr = best_last_state

    # trace backwards from the last position to the first
    for col in range(seq_len - 1, 0, -1):
        prev = viterbi_trace_matrix[curr][col]
        path.append(prev)
        curr = prev

    path.reverse()

    # map state indices back to their string labels
    state_path = "".join([id2state[s] for s in path])
    return state_path


result_path = traceback(viterbi_value_matrix, viterbi_trace_matrix)
print("Most probable state path:", result_path)
print("Query sequence:          ", query_sequence)


Best log probability: -40.06396
Most probable state path: EEEEEEEEEEEEEEEEEEEEEEEEEE
Query sequence:           CTTCATGTGAAAGCAGACGTAAGTCA
